In [50]:
'''Hohmann Transfer Delta-V Calculator
====================================
Computes the two delta-v burns required for a Hohmann transfer
between two circular (or elliptical) orbits around a primary body.

Physics background
------------------
Vis-viva equation  (gives orbital speed at any point on an ellipse):
    v = sqrt( mu * (2/r - 1/a) )

where:
    mu = G*M   gravitational parameter of the primary  [m³/s²]
    r  = distance from primary focus to the body      [m]
    a  = semi-major axis of the current orbit         [m]

For a CIRCULAR orbit of radius r:
    a = r  →  v_circ = sqrt(mu / r)

Hohmann transfer between two circular orbits r1 → r2
------------------------------------------------------
Transfer ellipse semi-major axis:
    a_t = (r1 + r2) / 2

Burn 1  (at periapsis of transfer ellipse, r = r1):
    v1_circ = sqrt(mu / r1)             current circular speed at r1
    v1_t    = sqrt(mu * (2/r1 - 1/a_t)) speed at periapsis of transfer ellipse
    dv1     = v1_t - v1_circ            (positive → prograde)

Burn 2  (at apoapsis of transfer ellipse, r = r2):
    v2_circ = sqrt(mu / r2)             target circular speed at r2
    v2_t    = sqrt(mu * (2/r2 - 1/a_t)) speed at apoapsis of transfer ellipse
    dv2     = v2_circ - v2_t            (positive → prograde to circularise)

Total delta-v:
    dv_total = |dv1| + |dv2|

General case (elliptical initial orbit)
----------------------------------------
If the spacecraft is already on an ellipse with semi-major axis a_init,
at distance r from the focus, its speed is given by vis-viva:
    v_current = sqrt(mu * (2/r - 1/a_init))

The transfer still departs from that point; burn 1 raises/lowers the orbit
so that the new ellipse has apoapsis (or periapsis) at r2.

Transfer ellipse with one apse at r and the other at r2:
    a_t = (r + r2) / 2
    v_t_at_r = sqrt(mu * (2/r - 1/a_t))
    dv1 = v_t_at_r - v_current

Burn 2 circularises at r2:
    v2_circ = sqrt(mu / r2)
    v2_t    = sqrt(mu * (2/r2 - 1/a_t))
    dv2     = v2_circ - v2_t
'''


'Hohmann Transfer Delta-V Calculator\n====================================\nComputes the two delta-v burns required for a Hohmann transfer\nbetween two circular (or elliptical) orbits around a primary body.\n\nPhysics background\n------------------\nVis-viva equation  (gives orbital speed at any point on an ellipse):\n    v = sqrt( mu * (2/r - 1/a) )\n\nwhere:\n    mu = G*M   gravitational parameter of the primary  [m³/s²]\n    r  = distance from primary focus to the body      [m]\n    a  = semi-major axis of the current orbit         [m]\n\nFor a CIRCULAR orbit of radius r:\n    a = r  →  v_circ = sqrt(mu / r)\n\nHohmann transfer between two circular orbits r1 → r2\n------------------------------------------------------\nTransfer ellipse semi-major axis:\n    a_t = (r1 + r2) / 2\n\nBurn 1  (at periapsis of transfer ellipse, r = r1):\n    v1_circ = sqrt(mu / r1)             current circular speed at r1\n    v1_t    = sqrt(mu * (2/r1 - 1/a_t)) speed at periapsis of transfer ellipse\n   

In [51]:

import math

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
G = 6.674_30e-11          # gravitational constant  [m³ kg⁻¹ s⁻²]
# ── Physical constants ─────────────────────────────────────────────────
M_earth  = 5.972_168e24     # [kg]
M_moon   = 7.342e22         # [kg]
R_earth  = 6_371_000.0      # mean radius [m]
R_moon   = 1_737_400.0      # mean radius [m]
d_moon   = 384_400_000.0    # mean Earth–Moon distance (centre–centre) [m]

mu_earth = G * M_earth      # 3.9860e14 m³/s²
mu_moon  = G * M_moon       # 4.9048e12 m³/s²

# ── Orbital radii (altitude above surface + body radius) ───────────────
alt_LEO  = 400_000.0        # LEO altitude [m]
alt_LLO  = 100_000.0        # Low Lunar Orbit altitude [m]

r_LEO  = R_earth + alt_LEO                  # 6 771 km from Earth centre
r_LLO  = R_moon  + alt_LLO                  # 1 837.4 km from Moon centre
r_moon_from_earth = d_moon                  # 384 400 km (Moon centre)


In [67]:
# ---------------------------------------------------------------------------
# Core functions
# ---------------------------------------------------------------------------

def vis_viva(mu: float, r: float, a: float) -> float:
    """
    Orbital speed at distance r on an ellipse with semi-major axis a.

    Parameters
    ----------
    mu : float  gravitational parameter G*M  [m³/s²]
    r  : float  current distance from focus  [m]
    a  : float  semi-major axis of orbit     [m]

    Returns
    -------
    float  speed in m/s
    """
    return math.sqrt(mu * (2.0 / r - 1.0 / a))


def hohmann_from_circular(mu: float, r1: float, r2: float) -> dict:
    """
    Classic Hohmann transfer between two CIRCULAR orbits.

    Parameters
    ----------
    mu : float  gravitational parameter  [m³/s²]
    r1 : float  radius of initial circular orbit   [m]
    r2 : float  radius of target  circular orbit   [m]

    Returns
    -------
    dict with keys:
        a_transfer   semi-major axis of transfer ellipse [m]
        v1_circ      speed on initial orbit  [m/s]
        v1_transfer  speed at periapsis of transfer ellipse [m/s]
        v2_transfer  speed at apoapsis  of transfer ellipse [m/s]
        v2_circ      speed on target orbit [m/s]
        dv1          delta-v of burn 1 [m/s]
        dv2          delta-v of burn 2 [m/s]
        dv_total     total delta-v  [m/s]
        tof_s        time of flight (half-period of transfer ellipse) [s]
        tof_h        time of flight [hours]
        tof_days     time of flight [days]
    """
    a_t = (r1 + r2) / 2.0

    v1c  = vis_viva(mu, r1, r1)    # circular at r1  (a=r1)
    v1t  = vis_viva(mu, r1, a_t)   # transfer ellipse at periapsis
    v2t  = vis_viva(mu, r2, a_t)   # transfer ellipse at apoapsis
    v2c  = vis_viva(mu, r2, r2)    # circular at r2

    dv1  = v1t - v1c
    dv2  = v2c - v2t

    # half-period of the transfer ellipse  T = pi * sqrt(a³/mu)
    tof  = math.pi * math.sqrt(a_t**3 / mu)

    return {
        "a_transfer":   a_t,
        "v1_circ":      v1c,
        "v1_transfer":  v1t,
        "v2_transfer":  v2t,
        "v2_circ":      v2c,
        "dv1":          dv1,
        "dv2":          dv2,
        "dv_total":     abs(dv1) + abs(dv2),
        "tof_s":        tof,
        "tof_h":        tof / 3600.0,
        "tof_days":     tof / 86400.0,
    }

def hohmann_to_ellipse(mu: float, r1: float, r2: float) -> dict:
    """
    Classic Hohmann transfer between two CIRCULAR orbits.

    Parameters
    ----------
    mu : float  gravitational parameter  [m³/s²]
    r1 : float  radius of initial circular orbit   [m]
    r2 : float  radius of target  circular orbit   [m]

    Returns
    -------
    dict with keys:
        a_transfer   semi-major axis of transfer ellipse [m]
        v1_circ      speed on initial orbit  [m/s]
        v1_transfer  speed at periapsis of transfer ellipse [m/s]
        v2_transfer  speed at apoapsis  of transfer ellipse [m/s]
        v2_circ      speed on target orbit [m/s]
        dv1          delta-v of burn 1 [m/s]
        dv2          delta-v of burn 2 [m/s]
        dv_total     total delta-v  [m/s]
        tof_s        time of flight (half-period of transfer ellipse) [s]
        tof_h        time of flight [hours]
        tof_days     time of flight [days]
    """
    a_t = (r1 + r2) / 2.0

    v1c  = vis_viva(mu, r1, r1)    # circular at r1  (a=r1)
    v1t  = vis_viva(mu, r1, a_t)   # transfer ellipse at periapsis
    

    dv1  = v1t - v1c
    

    # half-period of the transfer ellipse  T = pi * sqrt(a³/mu)
    tof  = math.pi * math.sqrt(a_t**3 / mu)

    return {
        "a_transfer":   a_t,
        "v1_circ":      v1c,
        "v1_transfer":  v1t,
        "dv1":          dv1,
        "dv_total":     abs(dv1),
        "tof_s":        tof,
        "tof_h":        tof / 3600.0,
        "tof_days":     tof / 86400.0,
    }

def hohmann_from_ellipse(
    mu: float,
    r: float,
    a_init: float,
    r_target: float,
) -> dict:
    """
    Hohmann-style transfer from an ELLIPTICAL orbit.

    The spacecraft is at distance r from the focus, on an orbit with
    semi-major axis a_init.  A single burn at this point sends it to
    a transfer ellipse whose opposite apse is at r_target, where a
    second burn circularises the orbit.

    Parameters
    ----------
    mu       : float  gravitational parameter  [m³/s²]
    r        : float  current distance from focus  [m]
    a_init   : float  semi-major axis of current orbit [m]
    r_target : float  target orbit radius (circular)  [m]

    Returns
    -------
    dict  (same keys as hohmann_from_circular, plus v_current)
    """
    a_t  = (r + r_target) / 2.0

    v_cur  = vis_viva(mu, r, a_init)   # current speed on initial ellipse
    v1t    = vis_viva(mu, r, a_t)      # speed after burn 1 on transfer ellipse
    v2t    = vis_viva(mu, r_target, a_t)
    v2c    = vis_viva(mu, r_target, r_target)

    dv1 = v1t  - v_cur
    dv2 = v2c  - v2t

    tof = math.pi * math.sqrt(a_t**3 / mu)

    return {
        "a_transfer":   a_t,
        "v_current":    v_cur,
        "v1_transfer":  v1t,
        "v2_transfer":  v2t,
        "v2_circ":      v2c,
        "dv1":          dv1,
        "dv2":          dv2,
        "dv_total":     abs(dv1) + abs(dv2),
        "tof_s":        tof,
        "tof_h":        tof / 3600.0,
        "tof_days":     tof / 86400.0,
    }



In [53]:
# ---------------------------------------------------------------------------
# Pretty printer
# ---------------------------------------------------------------------------

def print_hohmann(label: str, r1_km: float, r2_km: float, result: dict) -> None:
    sep = "=" * 62
    print(f"\n{sep}")
    print(f"  {label}")
    print(f"  r1 = {r1_km:>12,.1f} km   →   r2 = {r2_km:>12,.1f} km")
    print(sep)
    print(f"  Transfer ellipse semi-major axis : "
          f"{result['a_transfer']/1e3:>14,.1f} km")
    print()
    if "v_current" in result:
        print(f"  v (initial, on ellipse)          : "
              f"{result['v_current']:>10.4f} m/s  "
              f"({result['v_current']/1e3:.4f} km/s)")
    else:
        print(f"  v1 circular (at r1)              : "
              f"{result['v1_circ']:>10.4f} m/s  "
              f"({result['v1_circ']/1e3:.4f} km/s)")
    print(f"  v1 transfer  (periapsis)         : "
          f"{result['v1_transfer']:>10.4f} m/s  "
          f"({result['v1_transfer']/1e3:.4f} km/s)")
    print(f"  Δv₁ (burn 1, prograde)          : "
          f"{result['dv1']:>+10.4f} m/s  "
          f"({result['dv1']/1e3:+.4f} km/s)")
    print()
    print(f"  v2 transfer  (apoapsis)          : "
          f"{result['v2_transfer']:>10.4f} m/s  "
          f"({result['v2_transfer']/1e3:.4f} km/s)")
    print(f"  v2 circular  (at r2)             : "
          f"{result['v2_circ']:>10.4f} m/s  "
          f"({result['v2_circ']/1e3:.4f} km/s)")
    print(f"  Δv₂ (burn 2, circularise)        : "
          f"{result['dv2']:>+10.4f} m/s  "
          f"({result['dv2']/1e3:+.4f} km/s)")
    print()
    print(f"  ── TOTAL Δv ──────────────────── : "
          f"{result['dv_total']:>10.4f} m/s  "
          f"({result['dv_total']/1e3:.4f} km/s)")
    print()
    print(f"  Transfer time                    : "
          f"{result['tof_s']:>12.1f} s")
    print(f"                                   : "
          f"{result['tof_h']:>12.4f} h")
    print(f"                                   : "
          f"{result['tof_days']:>12.4f} days")
    print(sep)



In [54]:
# ---------------------------------------------------------------------------
# Application: Earth–Moon system
# ---------------------------------------------------------------------------

#def main() -> None:
print("\n" + "=" * 62)
print("  EARTH–MOON HOHMANN TRANSFER ANALYSIS")
print("  All distances from centre of primary body")
print("=" * 62)
print(f"\n  G              = {G:.6e}  m³ kg⁻¹ s⁻²")
print(f"  M_earth        = {M_earth:.6e}  kg")
print(f"  mu_earth (G·M) = {mu_earth:.6e}  m³/s²")
print(f"  M_moon         = {M_moon:.6e}  kg")
print(f"  mu_moon  (G·M) = {mu_moon:.6e}  m³/s²")



  EARTH–MOON HOHMANN TRANSFER ANALYSIS
  All distances from centre of primary body

  G              = 6.674300e-11  m³ kg⁻¹ s⁻²
  M_earth        = 5.972168e+24  kg
  mu_earth (G·M) = 3.986004e+14  m³/s²
  M_moon         = 7.342000e+22  kg
  mu_moon  (G·M) = 4.900271e+12  m³/s²


In [55]:
# ── Case 1: LEO → Lunar distance (Earth-centred, circular → circular) ──
# This gives the classic TLI + LOI picture but treating both orbits
# as circular around Earth (approximation: ignores Moon's gravity).
res1 = hohmann_from_circular(mu_earth, r_LEO, r_moon_from_earth)
print_hohmann(
    "CASE 1 · LEO → Moon distance  (Earth-centred, 2-burn Hohmann)",
    r_LEO / 1e3, r_moon_from_earth / 1e3, res1
)



  CASE 1 · LEO → Moon distance  (Earth-centred, 2-burn Hohmann)
  r1 =      6,771.0 km   →   r2 =    384,400.0 km
  Transfer ellipse semi-major axis :      195,585.5 km

  v1 circular (at r1)              :  7672.5983 m/s  (7.6726 km/s)
  v1 transfer  (periapsis)         : 10756.3723 m/s  (10.7564 km/s)
  Δv₁ (burn 1, prograde)          : +3083.7740 m/s  (+3.0838 km/s)

  v2 transfer  (apoapsis)          :   189.4677 m/s  (0.1895 km/s)
  v2 circular  (at r2)             :  1018.3034 m/s  (1.0183 km/s)
  Δv₂ (burn 2, circularise)        :  +828.8356 m/s  (+0.8288 km/s)

  ── TOTAL Δv ──────────────────── :  3912.6096 m/s  (3.9126 km/s)

  Transfer time                    :     430413.6 s
                                   :     119.5593 h
                                   :       4.9816 days


In [70]:
# Circular LEO to elliptical GEO
r_GEO   = 42_164_000.0

res_circ_to_ellipse = hohmann_to_ellipse(mu_earth, r_LEO, r_GEO)
res_circ_to_ellipse

{'a_transfer': 24467500.0,
 'v1_circ': 7672.598331010027,
 'v1_transfer': 10072.066090240625,
 'dv1': 2399.467759230598,
 'dv_total': 2399.467759230598,
 'tof_s': 19044.316842994354,
 'tof_h': 5.290088011942876,
 'tof_days': 0.22042033383095316}

In [56]:
# ── Case 2: Departure from an elliptical parking orbit ─────────────────
# Suppose the spacecraft is already in a highly-elliptical orbit:
# periapsis at LEO radius, apoapsis at GEO radius (~42 164 km).
# a_park = (r_LEO + r_GEO) / 2
r_GEO   = 42_164_000.0
a_park  = (r_LEO + r_GEO) / 2.0
# At periapsis of this parking ellipse the spacecraft fires to reach Moon.
res2 = hohmann_from_ellipse(mu_earth, r_LEO, a_park, r_moon_from_earth)
#print(res2)
print_hohmann(
    "CASE 2 · Elliptical park orbit → Moon  (burn at periapsis)",
    r_LEO / 1e3, r_moon_from_earth / 1e3, res2
)



  CASE 2 · Elliptical park orbit → Moon  (burn at periapsis)
  r1 =      6,771.0 km   →   r2 =    384,400.0 km
  Transfer ellipse semi-major axis :      195,585.5 km

  v (initial, on ellipse)          : 10072.0661 m/s  (10.0721 km/s)
  v1 transfer  (periapsis)         : 10756.3723 m/s  (10.7564 km/s)
  Δv₁ (burn 1, prograde)          :  +684.3062 m/s  (+0.6843 km/s)

  v2 transfer  (apoapsis)          :   189.4677 m/s  (0.1895 km/s)
  v2 circular  (at r2)             :  1018.3034 m/s  (1.0183 km/s)
  Δv₂ (burn 2, circularise)        :  +828.8356 m/s  (+0.8288 km/s)

  ── TOTAL Δv ──────────────────── :  1513.1418 m/s  (1.5131 km/s)

  Transfer time                    :     430413.6 s
                                   :     119.5593 h
                                   :       4.9816 days


In [57]:
res1, res2

({'a_transfer': 195585500.0,
  'v1_circ': 7672.598331010027,
  'v1_transfer': 10756.372288994064,
  'v2_transfer': 189.46773352960196,
  'v2_circ': 1018.30336851185,
  'dv1': 3083.7739579840363,
  'dv2': 828.8356349822479,
  'dv_total': 3912.6095929662843,
  'tof_s': 430413.59924005246,
  'tof_h': 119.5593331222368,
  'tof_days': 4.981638880093199},
 {'a_transfer': 195585500.0,
  'v_current': 10072.066090240625,
  'v1_transfer': 10756.372288994064,
  'v2_transfer': 189.46773352960196,
  'v2_circ': 1018.30336851185,
  'dv1': 684.3061987534384,
  'dv2': 828.8356349822479,
  'dv_total': 1513.1418337356863,
  'tof_s': 430413.59924005246,
  'tof_h': 119.5593331222368,
  'tof_days': 4.981638880093199})

In [58]:
'circular LEO', res1['dv_total'], 'ellipse GEO', res2['dv_total']

('circular LEO', 3912.6095929662843, 'ellipse GEO', 1513.1418337356863)

In [ ]:
# difference between circular LEO and elliptical GEO cases
diff = res1['dv_total'] - res2['dv_total']
print('Gain en dv (ellipse vs circular):', diff)
# dv to go elliptical GEO instead of circular LEO is the difference in dv between the two cases
diff, res_circ_to_ellipse['dv_total']

Gain en dv (ellipse vs circular): 2399.467759230598


(2399.467759230598, 2399.467759230598)

In [60]:
# ── Case 3: LLO → lunar surface (Moon-centred) ─────────────────────────
# Once captured in LLO, descend to the lunar surface.
# "Target orbit radius" = Moon's radius (surface).
res3 = hohmann_from_circular(mu_moon, r_LLO, R_moon)
print_hohmann(
    "CASE 3 · LLO → Lunar surface  (Moon-centred descent)",
    r_LLO / 1e3, R_moon / 1e3, res3
)



  CASE 3 · LLO → Lunar surface  (Moon-centred descent)
  r1 =      1,837.4 km   →   r2 =      1,737.4 km
  Transfer ellipse semi-major axis :        1,787.4 km

  v1 circular (at r1)              :  1633.0828 m/s  (1.6331 km/s)
  v1 transfer  (periapsis)         :  1610.0792 m/s  (1.6101 km/s)
  Δv₁ (burn 1, prograde)          :   -23.0036 m/s  (-0.0230 km/s)

  v2 transfer  (apoapsis)          :  1702.7509 m/s  (1.7028 km/s)
  v2 circular  (at r2)             :  1679.4232 m/s  (1.6794 km/s)
  Δv₂ (burn 2, circularise)        :   -23.3277 m/s  (-0.0233 km/s)

  ── TOTAL Δv ──────────────────── :    46.3313 m/s  (0.0463 km/s)

  Transfer time                    :       3391.3 s
                                   :       0.9420 h
                                   :       0.0393 days


In [61]:
# ── Case 4: Lunar surface → LLO  (ascent, reverse of Case 3) ──────────
res4 = hohmann_from_circular(mu_moon, R_moon, r_LLO)
print_hohmann(
    "CASE 4 · Lunar surface → LLO  (ascent burn)",
    R_moon / 1e3, r_LLO / 1e3, res4
)



  CASE 4 · Lunar surface → LLO  (ascent burn)
  r1 =      1,737.4 km   →   r2 =      1,837.4 km
  Transfer ellipse semi-major axis :        1,787.4 km

  v1 circular (at r1)              :  1679.4232 m/s  (1.6794 km/s)
  v1 transfer  (periapsis)         :  1702.7509 m/s  (1.7028 km/s)
  Δv₁ (burn 1, prograde)          :   +23.3277 m/s  (+0.0233 km/s)

  v2 transfer  (apoapsis)          :  1610.0792 m/s  (1.6101 km/s)
  v2 circular  (at r2)             :  1633.0828 m/s  (1.6331 km/s)
  Δv₂ (burn 2, circularise)        :   +23.0036 m/s  (+0.0230 km/s)

  ── TOTAL Δv ──────────────────── :    46.3313 m/s  (0.0463 km/s)

  Transfer time                    :       3391.3 s
                                   :       0.9420 h
                                   :       0.0393 days


In [62]:
# ── Case 5: Full mission Δv budget summary ─────────────────────────────
print("\n" + "=" * 62)
print("  FULL MISSION Δv BUDGET  (Earth surface excluded)")
print("=" * 62)


# choose 'circular' or 'ellipse' for TLI/LOI
init = 'ellipse'  
#init = 'circular'  
res = res1 if init=='circular' else res2

missions = [
    ("TLI  LEO → Moon SOI (Hohmann)",  res["dv1"]),
    ("LOI  Moon SOI → LLO",            res["dv2"]),
    ("PDI  LLO → Lunar surface",       res3["dv_total"]),
    ("Asc  Lunar surface → LLO",       res4["dv_total"]),
    ("TEI  LLO → Earth (reverse TLI)", res["dv1"]),   # symmetric
    ("EDL  Earth reentry (aerobrake)", 0.0),
]
total = 0.0
for name, dv in missions:
    total += abs(dv)
    note = "  ← aerobraking saves ~3 km/s" if dv == 0.0 else ""
    print(f"  {name:<38} {abs(dv)/1e3:>7.4f} km/s{note}")
print(f"  {'TOTAL (propulsive)':<38} {total/1e3:>7.4f} km/s")
print("=" * 62 + "\n")




  FULL MISSION Δv BUDGET  (Earth surface excluded)
  TLI  LEO → Moon SOI (Hohmann)           0.6843 km/s
  LOI  Moon SOI → LLO                     0.8288 km/s
  PDI  LLO → Lunar surface                0.0463 km/s
  Asc  Lunar surface → LLO                0.0463 km/s
  TEI  LLO → Earth (reverse TLI)          0.6843 km/s
  EDL  Earth reentry (aerobrake)          0.0000 km/s  ← aerobraking saves ~3 km/s
  TOTAL (propulsive)                      2.2901 km/s



In [63]:
'''
if __name__ == "__main__":
    main()
'''

'\nif __name__ == "__main__":\n    main()\n'